# Project Overview: Viet Nam Education & Economy (2011-2024)

## 1. Dataset Overview
This dataset is a specialized extract from the World Bank’s World Development Indicators (WDI). It focuses on the intersection of Viet Nam's economic growth and its educational system's performance. The objective is to facilitate data visualization and correlation analysis to determine how national wealth, income distribution, and government spending influence literacy, enrollment, and long-term academic attainment.

## 2. Data Structure
| Attribute | Details |
| :--- | :--- |
| **Total Variables** | 92 |
| **Time Frame** | 2011 - 2024 |
| **Regional Focus** | Viet Nam |
| **Key Categories** | Economics (GDP/GNI), Poverty & Equity, Education Quality, Gender Parity, and Attainment Levels. |
| **Missing Data Handling** | Usage of Interpolation and Omission for missing data |
| **Primary Units** | %, Constant 2015 US$, Ratios, and Numerical Counts. |

## Banana

![alt text](animation_frame_56-61.gif)


![](animation_frame_0-80.gif)

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [8]:


# 1. Define the subfolder name
subfolder = "data/"

# 2. Add the subfolder prefix to each filename
raw_filenames = [
    "2011-2017.csv", "2011-2018.csv", "2011-2021.csv", 
    "2011-2022-even.csv", "2011-2022.csv", "2011-2023.csv", 
    "2011-2024.csv", "2013-2018.csv", "2013-2024.csv", 
    "2014-2018.csv", "2014-2023.csv", "2014-2024.csv", 
    "2015-2022.csv", "2019-2023.csv"
]

file_list_with_path = [subfolder + f for f in raw_filenames]

class VietnamDataBody:
    def __init__(self, file_paths):
        self.file_paths = file_paths
        self.full_df = pd.DataFrame()
        self.data_map = {}
        self._build_data_body()

    def _build_data_body(self):
        all_frames = []
        for path in self.file_paths:
            try:
                # Pandas handles the path string directly
                df = pd.read_csv(path, skipfooter=5, engine='python')
                
                # Cleaning logic (same as before)
                df.columns = [c.split(' [')[0] if ' [' in c else c for c in df.columns]
                df = df.drop(columns=['Country Name', 'Country Code', 'Series Code'], errors='ignore')
                
                all_frames.append(df)
            except FileNotFoundError:
                print(f"Error: Could not find file at {path}. Check your subfolder name.")
            except Exception as e:
                print(f"Error processing {path}: {e}")

        if all_frames:
            combined = pd.concat(all_frames, axis=0, ignore_index=True)
            self.full_df = combined.groupby('Series Name').first()
            self.full_df = self.full_df.replace('..', np.nan).apply(pd.to_numeric)
            self.data_map = self.full_df.to_dict(orient='index')

    def get(self, keyword):
        matches = [k for k in self.data_map.keys() if keyword.lower() in k.lower()]
        if not matches: return None
        return self.data_map[matches[0]] if len(matches) == 1 else {m: self.data_map[m] for m in matches}

# Initialize the data body
data_body = VietnamDataBody(file_list_with_path)

# Quick check
print("Data loaded successfully. Total indicators:", len(data_body.full_df))

Data loaded successfully. Total indicators: 91


In [ ]:
def extract_indicator_metadata(data_body):
    metadata_list = []
    
    # Iterate through each row (Indicator) in the master DataFrame
    for indicator_name, row in data_body.full_df.iterrows():
        # 1. Get valid (non-NaN) values for calculation
        valid_values = row.dropna()
        
        # 2. Determine Year Range (e.g., "2011-2024")
        if not valid_values.empty:
            years = valid_values.index.astype(int)
            year_range = f"{years.min()}-{years.max()}"
            
            # 3. Calculate Stats
            mean_val = np.round(valid_values.mean(), 2)
            std_val = np.round(valid_values.std(), 2)
            
            # 4. Define Attributes
            # WDI data is typically Quantitative. 
            # We check if it's an integer-like count or a float percentage/ratio.
            if "index" in indicator_name.lower() or "ratio" in indicator_name.lower() or "%" in indicator_name:
                attr_type = "Quantitative (Continuous)"
            else:
                attr_type = "Quantitative (Discrete/Count)"
                
            dtype = "Real Value"
        else:
            year_range = "N/A"
            mean_val = "N/A"
            std_val = "N/A"
            attr_type = "N/A"
            dtype = "N/A"

        # Append data to list for the final table
        metadata_list.append({
            "Data Name": indicator_name,
            "Data Type": dtype,
            "Attribute": attr_type,
            "Year Range": year_range,
            "Mean": mean_val,
            "Std Dev": std_val
        })

    # Convert to a DataFrame for easy viewing/copying
    metadata_df = pd.DataFrame(metadata_list)
    return metadata_df

# --- Execution ---
# Assumes 'data_body' was initialized in the previous step
summary_table = extract_indicator_metadata(data_body)

# Display the first few rows to verify
print(summary_table.head(10))

# To get the full list:
# print(summary_table.to_string())

                                           Data Name   Data Type  \
0     Adjusted net national income (annual % growth)  Real Value   
1   Adjusted net national income (constant 2015 US$)  Real Value   
2  Adjusted net national income per capita (const...  Real Value   
3  Adjusted net savings, excluding particulate em...  Real Value   
4  Adjusted net savings, including particulate em...  Real Value   
5  Adolescents out of school (% of lower secondar...  Real Value   
6  Adolescents out of school, female (% of female...  Real Value   
7  Adolescents out of school, male (% of male low...  Real Value   
8   Children out of school (% of primary school age)  Real Value   
9                    Children out of school, primary  Real Value   

                       Attribute Year Range          Mean       Std Dev  
0      Quantitative (Continuous)  2011-2021  6.670000e+00  2.730000e+00  
1  Quantitative (Discrete/Count)  2011-2021  2.157671e+11  4.745323e+10  
2  Quantitative (Discrete/Cou

| Data Name | Data Type | Attribute	| Year Range | Mean	| Std Dev |
| :--- | :--- | :--- | :--- | :--- | :--- |
| Adjusted net national income (annual % growth) | Real Value | Quantitative (Continuous) | 2011-2021 | 6.67 | 2.73 |
| Adjusted net national income (constant 2015 US$) | Real Value | Quantitative (Discrete) | 2011-2021 | 215.77B | 47.45B |
| Adjusted net national income per capita (constant 2015 US$) | Real Value | Quantitative (Discrete) | 2011-2021 | 2,284.08 | 418.97 |
| Adjusted net savings, excluding emissions (% of GNI) | Real Value | Quantitative (Continuous) | 2011-2021 | 18.55 | 1.39 |
| Adjusted net savings, including emissions (% of GNI) | Real Value | Quantitative (Continuous) | 2011-2021 | 18.20 | 1.40 |
| Adolescents out of school (% of secondary age) | Real Value | Quantitative (Continuous) | 2014-2023 | 6.17 | 2.71 |
| Adolescents out of school, female (%) | Real Value | Quantitative (Continuous) | 2014-2023 | 7.17 | 2.12 |
| Adolescents out of school, male (%) | Real Value | Quantitative (Continuous) | 2014-2023 | 5.20 | 3.34 |
| Children out of school (% of primary age) | Real Value | Quantitative (Continuous) | 2014-2024 | 1.27 | 0.72 |
| Children out of school, primary (count) | Real Value | Quantitative (Discrete) | 2014-2024 | 105,716.5 | 63,123.4 |
| Compulsory education, duration (years) | Real Value | Quantitative (Continuous) | 2011-2024 | 9.71 | 0.47 |
| Ed. Attainment: Doctoral, female (%) | Real Value | Quantitative (Continuous) | 2019-2023 | 0.04 | 0.01 |
| Ed. Attainment: Doctoral, male (%) | Real Value | Quantitative (Continuous) | 2019-2023 | 0.10 | 0.01 |
| Ed. Attainment: Bachelor's, total (%) | Real Value | Quantitative (Continuous) | 2019-2023 | 10.74 | 0.62 |
| Ed. Attainment: Master's, total (%) | Real Value | Quantitative (Continuous) | 2019-2023 | 0.68 | 0.04 |
| Ed. Attainment: Lower Secondary, total (%) | Real Value | Quantitative (Continuous) | 2019-2023 | 64.77 | 1.75 |
| Ed. Attainment: Primary, total (%) | Real Value | Quantitative (Continuous) | 2011-2023 | 82.60 | 3.52 |
| Ed. Attainment: Upper Secondary, total (%) | Real Value | Quantitative (Continuous) | 2011-2023 | 31.99 | 4.99 |
| GDP (constant 2015 US$) | Real Value | Quantitative (Discrete) | 2011-2024 | 285.63B | 69.60B |
| GDP growth (annual %) | Real Value | Quantitative (Continuous) | 2011-2024 | 6.10 | 1.69 |
| GDP per capita (constant 2015 US$) | Real Value | Quantitative (Discrete) | 2011-2024 | 2,974.24 | 601.11 |
| GNI growth (annual %) | Real Value | Quantitative (Continuous) | 2011-2024 | 6.58 | 2.95 |
| Gini index | Real Value | Quantitative (Continuous) | 2012-2022 | 35.72 | 0.69 |
| Gov expenditure on education (% of GDP) | Real Value | Quantitative (Continuous) | 2011-2022 | 3.50 | 0.52 |
| Gov expenditure on education (% of gov exp) | Real Value | Quantitative (Continuous) | 2011-2022 | 16.20 | 1.45 |
| Income share held by lowest 20% | Real Value | Quantitative (Continuous) | 2012-2022 | 6.87 | 0.21 |
| Income share held by highest 20% | Real Value | Quantitative (Continuous) | 2012-2022 | 42.95 | 0.59 |
| Multidimensional poverty headcount (%) | Real Value | Quantitative (Continuous) | 2012-2022 | 3.32 | 1.40 |
| Over-age students, primary (%) | Real Value | Quantitative (Continuous) | 2011-2018 | 2.22 | 0.90 |
| Poverty headcount ratio at $3.00/day (%) | Real Value | Quantitative (Continuous) | 2012-2022 | 2.45 | 0.96 |
| Poverty headcount ratio at $8.30/day (%) | Real Value | Quantitative (Continuous) | 2012-2022 | 29.50 | 9.31 |
| Primary education, pupils (count) | Real Value | Quantitative (Discrete) | 2011-2024 | 8.10M | 786,221 |
| Pupil-teacher ratio, primary | Real Value | Quantitative (Continuous) | 2011-2018 | 19.49 | 0.42 |
| Repeaters, primary, total (%) | Real Value | Quantitative (Continuous) | 2011-2018 | 1.03 | 0.25 |
| School enrollment, preprimary (% gross) | Real Value | Quantitative (Continuous) | 2011-2021 | 87.96 | 8.21 |
| School enrollment, primary (% gross) | Real Value | Quantitative (Continuous) | 2011-2021 | 103.29 | 1.25 |
| School enrollment, tertiary (% gross) | Real Value | Quantitative (Continuous) | 2011-2017 | 27.62 | 2.37 |
| Trained teachers in primary education (%) | Real Value | Quantitative (Continuous) | 2014-2024 | 93.55 | 9.17 |
| Trained teachers in preprimary, male (%) | Real Value | Quantitative (Continuous) | 2014-2024 | 82.48 | 16.33 |